# 🎯 Customer 360: RFM Segmentation & Cohort Retention Modeling
**Project**: Global E-Commerce Analytics  
**Author**: Tushar Joshi | Data Analyst Portfolio  

---
### 📌 Notebook Objectives
1. Calculate Recency, Frequency, and Monetary (RFM) metrics for every unique customer.
2. Assign quintile RFM scores (1-5) and segment customers into strategic personas (Champions, At Risk, Lost, etc.).
3. Build customer cohort groups based on initial signup month.
4. Generate a Month-by-Month Cohort Retention Heatmap matrix to evaluate retention decay.
5. Formulate actionable data-driven marketing & customer success strategies.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans

sns.set_theme(style="white", palette="muted")
plt.rcParams["figure.figsize"] = (12, 6)

# Load cleaned dataset
df = pd.read_csv("../data/cleaned_ecommerce_data.csv")
df["order_date"] = pd.to_datetime(df["order_date"])
print(f"Loaded {len(df)} transactions.")

## 1. Recency, Frequency, & Monetary (RFM) Calculation

In [ ]:
snapshot_date = df["order_date"].max() + pd.Timedelta(days=1)

rfm = df.groupby("customer_id").agg({
    "order_date": lambda x: (snapshot_date - x.max()).days,
    "order_id": "nunique",
    "total_revenue": "sum",
    "region": "first"
}).reset_index()

rfm.columns = ["customer_id", "recency_days", "frequency", "monetary", "region"]
rfm.head()

## 2. RFM Quintile Scoring & Segment Mapping

In [ ]:
rfm["R_Score"] = pd.qcut(rfm["recency_days"].rank(method='first', ascending=False), q=5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["F_Score"] = pd.qcut(rfm["frequency"].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["M_Score"] = pd.qcut(rfm["monetary"].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5]).astype(int)

def assign_segment(row):
    r, f, m = row["R_Score"], row["F_Score"], row["M_Score"]
    if r >= 4 and f >= 4 and m >= 4:
        return "Champions"
    elif r >= 3 and f >= 3:
        return "Loyal Customers"
    elif r >= 3 and f < 3 and m >= 3:
        return "Potential Loyalists"
    elif r >= 4 and f == 1:
        return "New Customers"
    elif r <= 2 and f >= 3:
        return "At Risk"
    elif r <= 2 and f <= 2 and m >= 3:
        return "Cant Lose Them"
    else:
        return "Lost / Hibernating"

rfm["Customer_Segment"] = rfm.apply(assign_segment, axis=1)

print("--- Customer Segment Distribution ---")
print(rfm["Customer_Segment"].value_counts())

## 3. Visualizing Customer Segments

In [ ]:
plt.figure(figsize=(12, 6))
seg_counts = rfm["Customer_Segment"].value_counts().reset_index()
sns.barplot(data=seg_counts, x="count", y="Customer_Segment", palette="Spectral")
plt.title("Customer Distribution Across RFM Personas", fontsize=14, fontweight="bold")
plt.xlabel("Number of Customers")
plt.ylabel("Segment")
plt.tight_layout()
plt.show()

## 4. Cohort Retention Matrix Analysis

In [ ]:
df["cohort_month"] = df.groupby("customer_id")["order_date"].transform("min").dt.to_period("M").astype(str)
df["order_month"] = df["order_date"].dt.to_period("M").astype(str)

def get_month_index(df):
    cohort_yr = pd.to_datetime(df["cohort_month"]).dt.year
    cohort_mo = pd.to_datetime(df["cohort_month"]).dt.month
    order_yr = pd.to_datetime(df["order_month"]).dt.year
    order_mo = pd.to_datetime(df["order_month"]).dt.month
    return (order_yr - cohort_yr) * 12 + (order_mo - cohort_mo)

df["cohort_index"] = get_month_index(df)

cohort_group = df.groupby(["cohort_month", "cohort_index"])["customer_id"].nunique().reset_index()
cohort_pivot = cohort_group.pivot(index="cohort_month", columns="cohort_index", values="customer_id")
cohort_size = cohort_pivot.iloc[:, 0]
retention_matrix = cohort_pivot.divide(cohort_size, axis=0).round(4) * 100

plt.figure(figsize=(16, 10))
sns.heatmap(retention_matrix, annot=True, fmt=".1f", cmap="YlGnBu", cbar_kws={'label': 'Retention Rate (%)'})
plt.title("Monthly Cohort Retention Heatmap (%)", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Months Since First Order")
plt.ylabel("Cohort Month")
plt.tight_layout()
plt.show()

## 5. Strategic Recommendations & Export

In [ ]:
# Save RFM segmentation data
rfm.to_csv("../data/rfm_segmented_customers.csv", index=False)
print("✅ Segmentation exported to '../data/rfm_segmented_customers.csv'")